In [ ]:
# Figure 2
# Exhaustive search results

In [ ]:
import math

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt


%matplotlib inline

In [ ]:
import sys
sys.path.append("../python/")
from distance_stats import read_data

In [ ]:
k = 3
n = 6
plots_dir = "../plots/"
results_path = "../../results/reproduce/exhaustive/"
m_vals = list(range(2, 21))

In [ ]:
# Count the total number of k-uniform hypergraphs with m hyperedges
possibilities = np.zeros(len(m_vals))
for col, num_edges in enumerate(m_vals):
    possible_hyperedges = math.comb(n, k)
    possibilities[col] = math.comb(possible_hyperedges, num_edges)

In [ ]:
# For every m, read the exhaustive search results files and
# get the number of unique projections, the total number of
# uniform twins, and the number of uniform isomorphism classes

# Total number of unique projections
total_projections = np.zeros(len(m_vals))

# Total k-uniform twins, including isomorphs
total_twins_uniform = np.zeros(len(m_vals))

# Total k-uniform isomorphism calsses
total_twin_iso_classes = np.zeros(len(m_vals))

for m in m_vals:
    filename = f"n-{n}_m-{m}_k-{k}_non-uniform_exhaustive_projections.csv"
    output_data = read_data(results_path + filename, xgi_hypergraphs=False)
    midx = m-min(m_vals)
    print(f"m: {m}, projections: {len(output_data['m'])}, midx: {midx}")
    if len(output_data["m"]) > 0:
        # Get the total number of projections, which is just the number of rows
        total_projections[midx] = len(output_data["m"])
        # For every sample
        for proj_idx in range(len(output_data["m"])):
            # Get filtered and unfiltered 3-uniform hypergraphs
            for size_dist in output_data["dist_results_dict"][proj_idx]:
                if sum([tup[1] for tup in size_dist]) != m:
                    continue
                sizes = [tup[0] for tup in size_dist]
                if (min(sizes) == 3) and (max(sizes) == 3):
                    total_twin_iso_classes[midx] += output_data["dist_results_dict"][proj_idx][size_dist]["filtered"]
                    total_twins_uniform[midx] += output_data["dist_results_dict"][proj_idx][size_dist]["unfiltered"]


In [ ]:
total_projections = np.array(total_projections)
total_twins_uniform = np.array(total_twins_uniform)
total_twin_iso_classes = np.array(total_twin_iso_classes)

In [ ]:
# For each unique projection in m=10, count the number of isomorphism classes
m = 10
filename = f"n-{n}_m-{m}_k-{k}_non-uniform_exhaustive_projections.csv"
output_data = read_data(results_path + filename, xgi_hypergraphs=False)
one_dist = []
two_dist = []
other_dist = []

for proj_idx in range(len(output_data["m"])):
    # Get filtered and unfiltered 3-uniform hypergraphs
    for size_dist, res in output_data["dist_results_dict"][proj_idx].items():
        if sum([tup[1] for tup in size_dist]) != m:
            continue

        sizes = [tup[0] for tup in size_dist]
        if (min(sizes) == 3) and (max(sizes) == 3):
            if res["unfiltered"] < 2:
                continue

            if res["filtered"] == 1:
                one_dist.append(res["unfiltered"])
            elif res["filtered"] == 2:
                two_dist.append(res["unfiltered"])
            else:
                other_dist.append(res["unfiltered"])

dist_length = max([max(one_dist), max(two_dist)])+1
b1 = np.zeros(dist_length)
b2 = np.zeros(dist_length)

for val in one_dist:
    b1[val] += 1

for val in two_dist:
    b2[val] += 1

# x positions for plotting
offset = 0.15
x_left = []
x_right = []
for val in range(dist_length):
    x_left.append(val-offset)
    x_right.append(val+offset)

In [ ]:
label_size = 20
tick_size = 18
legend_font = 18
labelpad = 3
mpl.rcParams["axes.labelpad"] = labelpad
mpl.rcParams["axes.labelsize"] = label_size
mpl.rcParams["legend.fontsize"] = legend_font
mpl.rcParams["xtick.labelsize"] = tick_size
mpl.rcParams["ytick.labelsize"] = tick_size

In [ ]:
text_label_x = 0.075
text_label_y = 0.975

In [ ]:
fig, axs = plt.subplots(2, 2, squeeze=False, constrained_layout=True, figsize=(10, 7))

mticks = list(range(0, 21, 2))
row = col = 0
axs[row][col].plot(m_vals, possibilities,
        label="Hypergraphs",
       c="C1"
       )
axs[row][col].set_ylabel("Hypergraphs")
axs[row][col].set_yscale("log")
axs[row][col].spines[['top', 'right']].set_visible(False)
axs[row][col].text(text_label_x, text_label_y, "(a)", size=label_size, fontweight="bold", horizontalalignment='center',
     verticalalignment='center', transform=axs[row][col].transAxes)
axs[row][col].set_xticks(mticks, labels=mticks)
axs[row][col].set_xlabel("$m$")

col = 1
axs[row][col].plot(m_vals, total_projections, label="Projections")
axs[row][col].set_ylabel("Unique Projections")
axs[row][col].set_xticks(mticks, labels=mticks)
axs[row][col].set_xlabel("$m$")
axs[row][col].spines[['top', 'right']].set_visible(False)
axs[row][col].text(text_label_x, text_label_y, "(b)", size=label_size, fontweight="bold", horizontalalignment='center',
     verticalalignment='center', transform=axs[row][col].transAxes)

row = 1
col = 0
y = (total_twins_uniform / total_projections) * 100
axs[row][col].plot(m_vals, y,
                   c="C2",
                  label="Total",
                  )
y = (total_twin_iso_classes / total_projections) * 100
axs[row][col].plot(m_vals, y,
                   c="C4",
                  label="Iso.\nClasses",
                  linestyle="dashed",
                  )
axs[row][col].set_ylabel("% Rel. to # Projections")
axs[row][col].set_xticks(mticks, labels=mticks)
axs[row][col].set_xlabel("$m$")
axs[row][col].spines[['right', 'top']].set_visible(False)
axs[row][col].text(text_label_x, text_label_y, "(c)", size=label_size, fontweight="bold", horizontalalignment='center',
     verticalalignment='center', transform=axs[row][col].transAxes)
axs[row][col].legend(ncols=1, frameon=False, loc=(0.6, 0.7))

col = 1

axs[row][col].bar(x_left, b1, width=0.275, color="rosybrown")
axs[row][col].bar(x_right, b2, width=0.275, color="dimgray", hatch="///", fill=False)
axs[row][col].set_xticks(list(range(0, dist_length+1, 2)));
axs[row][col].spines[['right', 'top']].set_visible(False)
axs[row][col].set_xlabel("Total # Twins")
axs[row][col].set_ylabel("Projections")
axs[row][col].legend(["1 Iso. Class", "2 Iso. Class"], frameon=False)
axs[row][col].text(text_label_x, text_label_y, "(d)", size=label_size, fontweight="bold", horizontalalignment='center',
     verticalalignment='center', transform=axs[row][col].transAxes)

fig.savefig(plots_dir + "exhaustive-combined.pdf", dpi=300)